# Encontro 4 — Importação, inspeção e compreensão inicial de dados com Pandas

> **Ideia central:** importar uma base é apenas o começo. Antes de limpar, transformar ou modelar, precisamos verificar o que recebemos e construir um diagnóstico inicial.

Este notebook foi planejado para ser seguido com autonomia, em Jupyter ou no **Google Colab**. Ele usa a base **Automobile (Imports-85)**, com 205 automóveis e 26 variáveis, disponibilizada pela UCI Machine Learning Repository.

**Ao final, você deverá conseguir:**

1. carregar um arquivo CSV com `pandas`;
2. verificar se a importação ocorreu corretamente;
3. reconhecer observações, variáveis, tipos e unidade de análise;
4. localizar valores ausentes e valores distintos;
5. propor uma variável-alvo e perguntas analíticas;
6. registrar um diagnóstico inicial — sem ainda limpar a base.


## Como usar no Google Colab

1. Abra o arquivo `.ipynb` no Colab.
2. Execute as células na ordem, usando **Shift + Enter**.
3. Se a pasta `dados/` estiver junto do notebook, ela será usada automaticamente.
4. Se os arquivos locais não estiverem disponíveis, o notebook baixa a versão bruta diretamente da UCI e aplica os nomes das colunas.

As caixas **Pare e pense** devem ser respondidas antes de abrir a sugestão de resposta.


## 1. O problema antes do código

Imagine que alguém enviou um arquivo chamado `automoveis.csv` e pediu uma análise de preços. Antes de calcular qualquer coisa, precisamos perguntar:

- O que representa cada linha?
- O que representa cada coluna?
- Há uma variável que pode funcionar como alvo?
- Os tipos inferidos fazem sentido?
- Como os valores ausentes foram registrados?
- A importação produziu a estrutura esperada?

Essas perguntas pertencem à etapa de **compreensão dos dados**.


## 2. Preparação do ambiente

Importaremos somente as bibliotecas necessárias para esta atividade.

In [ ]:
from pathlib import Path
from io import StringIO
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")

print("Versão do Pandas:", pd.__version__)


### Nomes das variáveis

O arquivo bruto da UCI não possui cabeçalho. Por isso, precisamos conhecer o esquema da base antes de carregá-lo. Os mesmos nomes também aparecem no arquivo `automoveis.csv` fornecido com este material.


In [ ]:
COLUNAS = [
    "symboling", "normalized-losses", "make", "fuel-type", "aspiration",
    "num-of-doors", "body-style", "drive-wheels", "engine-location",
    "wheel-base", "length", "width", "height", "curb-weight",
    "engine-type", "num-of-cylinders", "engine-size", "fuel-system",
    "bore", "stroke", "compression-ratio", "horsepower", "peak-rpm",
    "city-mpg", "highway-mpg", "price"
]

len(COLUNAS)


## 3. Carregamento da base

A célula abaixo procura primeiro o arquivo local. No Colab, se ele não existir, usa a URL oficial da UCI.

In [ ]:
caminhos_locais = [
    Path("dados/automoveis.csv"),
    Path("automoveis.csv"),
]

caminho_encontrado = next((c for c in caminhos_locais if c.exists()), None)

if caminho_encontrado is not None:
    df = pd.read_csv(caminho_encontrado, na_values="?")
    origem_dados = str(caminho_encontrado)
else:
    url_uci = (
        "https://archive.ics.uci.edu/ml/"
        "machine-learning-databases/autos/imports-85.data"
    )
    df = pd.read_csv(url_uci, names=COLUNAS, na_values="?")
    origem_dados = url_uci

print("Origem:", origem_dados)
print("Base carregada com sucesso.")


### Primeira validação

Conhecemos duas expectativas fornecidas pelo dicionário da UCI: **205 observações** e **26 variáveis**. Uma validação simples impede que uma importação estruturalmente errada passe despercebida.


In [ ]:
assert df.shape == (205, 26), (
    f"Esperávamos 205 linhas e 26 colunas, mas obtivemos {df.shape}."
)

print("Dimensões confirmadas:", df.shape)


<div style="border-left: 4px solid #3D8DFF; padding-left: 12px">
<strong>Pare e pense:</strong> o código ter executado sem erro seria suficiente para concluir que a importação está correta? Por quê?
</div>

<details><summary>Sugestão de resposta</summary>

Não. Um separador incorreto, por exemplo, pode gerar um DataFrame com uma única coluna sem produzir exceção. Precisamos verificar dimensões, nomes, tipos e exemplos de registros.

</details>


## 4. Inspeção visual dos registros

`head()`, `tail()` e `sample()` mostram partes diferentes da base. Nenhum deles, isoladamente, representa a base inteira.

In [ ]:
df.head()


In [ ]:
df.tail(3)


In [ ]:
df.sample(5, random_state=42)


<div style="border-left: 4px solid #3D8DFF; padding-left: 12px">
<strong>Pare e pense:</strong> qual é a unidade de análise desta base?
</div>

<details><summary>Sugestão de resposta</summary>

Cada linha representa um automóvel descrito por características técnicas, de consumo, risco de seguro e preço. Não há uma coluna de identificador único explícita.

</details>


## 5. Dimensões e nomes das variáveis

`shape` devolve uma tupla no formato `(linhas, colunas)`. `columns` apresenta os nomes recebidos pelo Pandas.

In [ ]:
numero_linhas, numero_colunas = df.shape

print("Número de observações:", numero_linhas)
print("Número de variáveis:", numero_colunas)
print("\nNomes das colunas:")
print(df.columns.tolist())


## 6. Tipos inferidos

`dtypes` mostra como o Pandas interpretou cada coluna. Isso é um **diagnóstico**, não uma garantia de que o tipo esteja semanticamente correto.

Uma variável numérica pode ser importada como `object` quando contém símbolos, textos ou valores ausentes representados de maneira inadequada.


In [ ]:
df.dtypes


<div style="border-left: 4px solid #3D8DFF; padding-left: 12px">
<strong>Pare e pense:</strong> uma coluna com números armazenados como texto é categórica ou é uma variável numérica com problema de qualidade?
</div>

<details><summary>Sugestão de resposta</summary>

O tipo computacional e o significado da variável são coisas diferentes. É preciso consultar exemplos e o dicionário de dados antes de classificar ou converter a coluna.

</details>


## 7. Visão geral com `info()`

`info()` reúne dimensões, nomes, contagem de valores não nulos, tipos e uso aproximado de memória.

In [ ]:
df.info()


## 8. Resumo estatístico

`describe()` responde perguntas como média, dispersão, mínimo, máximo e quartis para colunas numéricas. Ele é apenas uma primeira visão — não substitui a análise exploratória.


In [ ]:
df.describe().T


Para observar também colunas categóricas, podemos usar `include='all'`. A tabela fica maior; use-a como consulta.

In [ ]:
df.describe(include='all').T


## 9. Valores ausentes

Na fonte original, algumas ausências são representadas por `?`. No carregamento usamos `na_values="?"`, informando ao Pandas que esse símbolo significa ausência.

Nesta aula, apenas **diagnosticaremos** as ausências. A decisão de remover, substituir ou investigar ficará para o próximo encontro.


In [ ]:
ausentes = df.isna().sum().sort_values(ascending=False)
ausentes[ausentes > 0]


In [ ]:
percentual_ausente = (
    df.isna().mean().mul(100).sort_values(ascending=False)
)

percentual_ausente[percentual_ausente > 0].round(2)


## 10. Quantidade de valores distintos

`nunique()` ajuda a encontrar colunas com baixa cardinalidade, possíveis categorias e possíveis identificadores.

In [ ]:
df.nunique(dropna=False).sort_values()


<div style="border-left: 4px solid #3D8DFF; padding-left: 12px">
<strong>Pare e pense:</strong> `make`, `fuel-type`, `horsepower` e `price` parecem ter o mesmo papel analítico?
</div>

<details><summary>Sugestão de resposta</summary>

Não. `make` e `fuel-type` são categorias; `horsepower` é uma medida numérica; `price` também é numérica e pode ser escolhida como variável-alvo em um problema de regressão.

</details>


## 11. Uma classificação inicial das variáveis

O código abaixo separa colunas pelo **tipo inferido pelo Pandas**. Depois, devemos complementar essa informação com o significado de negócio.


In [ ]:
variaveis_numericas = df.select_dtypes(include="number").columns.tolist()
variaveis_categoricas = df.select_dtypes(exclude="number").columns.tolist()

print("Numéricas:", variaveis_numericas)
print("\nCategóricas ou textuais:", variaveis_categoricas)


### Possível variável-alvo

Se a pergunta for **“quais características ajudam a explicar o preço de um automóvel?”**, então `price` é uma possível variável-alvo. Essa escolha depende do problema — não é uma propriedade fixa do arquivo.


In [ ]:
alvo = "price"
atributos = [coluna for coluna in df.columns if coluna != alvo]

print("Alvo possível:", alvo)
print("Quantidade de atributos candidatos:", len(atributos))


## 12. Experimento: executou sem erro, mas importou errado

O material inclui a mesma base com separador `;`. Primeiro, vamos lê-la usando `,` de propósito.


In [ ]:
candidatos_ponto_e_virgula = [
    Path("dados/automoveis_ponto_e_virgula.csv"),
    Path("automoveis_ponto_e_virgula.csv"),
]

caminho_sep = next(
    (c for c in candidatos_ponto_e_virgula if c.exists()),
    None,
)

if caminho_sep is not None:
    fonte_sep = caminho_sep
else:
    # Fallback para o Colab quando apenas o notebook foi aberto.
    fonte_sep = StringIO(df.to_csv(index=False, sep=";"))

df_errado = pd.read_csv(fonte_sep, sep=",")

print("Dimensões da importação errada:", df_errado.shape)
df_errado.head(2)


O código executou, mas todas as variáveis ficaram reunidas em uma única coluna. Agora usamos o separador correto.


In [ ]:
if hasattr(fonte_sep, "seek"):
    fonte_sep.seek(0)

df_correto = pd.read_csv(fonte_sep, sep=";", na_values="?")

print("Dimensões da importação correta:", df_correto.shape)
df_correto.head(2)


### Outros parâmetros frequentes

```python
pd.read_csv("dados.csv", sep=";", encoding="utf-8")
pd.read_csv("dados_sem_cabecalho.csv", header=None, names=COLUNAS)
pd.read_excel("dados.xlsx")
pd.read_json("dados.json")
```

Erros e sintomas úteis para investigar:

- `FileNotFoundError`: caminho inexistente ou pasta atual diferente;
- uma única coluna: separador provavelmente incorreto;
- acentos corrompidos ou `UnicodeDecodeError`: codificação incompatível;
- primeira observação usada como nomes: configuração de cabeçalho incorreta;
- números como `object`: símbolos, textos ou ausências não reconhecidas.


## 13. Dados também podem vir de bancos

O exemplo abaixo cria um banco SQLite temporário apenas para mostrar a ponte entre SQL e Pandas. Não estudaremos SQL em profundidade nesta aula.


In [ ]:
conexao = sqlite3.connect(":memory:")
df.to_sql("automoveis", conexao, index=False, if_exists="replace")

consulta = (
    'SELECT make, "body-style", horsepower, price '
    'FROM automoveis LIMIT 5'
)

amostra_banco = pd.read_sql_query(consulta, conexao)
amostra_banco


In [ ]:
conexao.close()


## 14. Prática orientada

Realize as tarefas abaixo usando `df`. Os blocos de solução vêm depois; tente responder antes de abri-los.

1. Exiba cinco registros aleatórios com resultado reproduzível.
2. Informe o número de linhas e colunas em uma frase.
3. Crie uma lista de colunas categóricas.
4. Mostre apenas as colunas que possuem valores ausentes.
5. Descubra quantos fabricantes distintos aparecem em `make`.
6. Identifique três variáveis numéricas e três categóricas.
7. Escolha uma possível variável-alvo e justifique em uma frase.
8. Escreva três perguntas analíticas possíveis.


In [ ]:
# Tarefa 1
# Sua resposta aqui

# Tarefa 2
# Sua resposta aqui

# Tarefa 3
# Sua resposta aqui

# Tarefa 4
# Sua resposta aqui

# Tarefa 5
# Sua resposta aqui


<details><summary>Soluções possíveis — tarefas 1 a 5</summary>

```python
# 1
df.sample(5, random_state=10)

# 2
print(f"A base possui {df.shape[0]} linhas e {df.shape[1]} colunas.")

# 3
categoricas = df.select_dtypes(exclude="number").columns.tolist()
categoricas

# 4
contagem_ausentes = df.isna().sum()
contagem_ausentes[contagem_ausentes > 0]

# 5
df["make"].nunique()
```

</details>


### Respostas conceituais — tarefas 6 a 8

Registre suas decisões como texto no notebook. Não existe uma única pergunta analítica correta, desde que ela seja clara e compatível com as variáveis disponíveis.

<details><summary>Exemplo de resposta</summary>

- Numéricas: `horsepower`, `engine-size`, `price`.
- Categóricas: `make`, `fuel-type`, `body-style`.
- Alvo possível: `price`, porque poderíamos investigar quais características técnicas ajudam a explicar o preço.
- Perguntas:
  1. O preço médio varia entre estilos de carroceria?
  2. Potência e tamanho do motor estão associados ao preço?
  3. Fabricantes diferentes apresentam padrões distintos de consumo na cidade?

</details>


## 15. Produto final: diagnóstico inicial da base

Vamos criar uma tabela-resumo por variável. Ela registra o que sabemos **antes** da limpeza.


In [ ]:
diagnostico = pd.DataFrame({
    "variavel": df.columns,
    "tipo_inferido": df.dtypes.astype(str).values,
    "valores_ausentes": df.isna().sum().values,
    "percentual_ausente": (df.isna().mean().mul(100).round(2)).values,
    "valores_distintos": df.nunique(dropna=False).values,
})

diagnostico


Exportaremos somente o diagnóstico. A base original permanece inalterada.

In [ ]:
diagnostico.to_csv("diagnostico_inicial_automoveis.csv", index=False)

print("Arquivo criado: diagnostico_inicial_automoveis.csv")


## Checklist da entrega

- [ ] bibliotecas importadas;
- [ ] base carregada;
- [ ] cinco registros exibidos;
- [ ] dimensões informadas;
- [ ] nomes e tipos inspecionados;
- [ ] resumo estatístico consultado;
- [ ] valores ausentes diagnosticados;
- [ ] unidade de análise descrita;
- [ ] possível variável-alvo justificada;
- [ ] três perguntas analíticas formuladas.


## Fechamento

> **Importar não é compreender.**  
> **Compreender não é limpar.**  
> **Limpar não é analisar.**

Neste encontro, a saída é um **diagnóstico inicial**. No próximo, esse diagnóstico orientará decisões sobre valores ausentes, formatação e qualidade dos dados.


## Fonte e licença

- Schlimmer, J. (1985). **Automobile** [Dataset]. UCI Machine Learning Repository. DOI: [10.24432/C5B01C](https://doi.org/10.24432/C5B01C).
- Página da base: <https://archive.ics.uci.edu/dataset/10/automobile>
- Licença: **Creative Commons Attribution 4.0 International (CC BY 4.0)**.

O roteiro didático foi alinhado ao **Module 1 — Importing Data Sets**, do curso *Data Analysis with Python*, IBM Data Science Professional Certificate.
